In [ ]:
import random

from distributed_processing.utils import fsclient

from distributed_processing.async_result import gather

from dotenv import load_dotenv
from os import getenv


In [ ]:
load_dotenv()
NS_PATH = getenv("NS_PATH")

In [3]:
import logging
logging.getLogger("distributed_processing").setLevel(logging.DEBUG)
#logging.getLogger("distributed_processing.filesystem_connector").setLevel(logging.DEBUG)

In [4]:
"""
fs_connector = FileSystemConnector(NS_PATH)
fs_connector.with_watchdog=True
fs_connector.pop_watchdog_timeout = 10

client = Client(DummySerializer(), fs_connector, check_registry="cache")
"""

client = fsclient(NS_PATH)

In [5]:
client.update_registry_cache()

In [6]:
y = client.rpc_async("add", [1, 0])

In [7]:
y.get()

1

In [8]:
client.check_registry ="always"

try:
    y = client.rpc_async("fake", [1, 0])
except Exception as e:
    print(e)

Method fake does not exist/is not available.


In [9]:
client.check_registry ="never" # use queue client.default_queue, by default "default"
client.set_default_queue("cola_1")

y = client.rpc_async("fake", [1, 0])

try:
    y.get()
except Exception as e:
    print(e)


Error -32601 : The method does not exist/is not available.

 NoneType: None



In [10]:
client.check_registry ="never"
client.set_default_queue("cola_1")

y = client.rpc_async("fake", [1, 0])

y.safe_get(default="Esto es un error")

'Esto es un error'

In [11]:
client.check_registry ="cache"

In [12]:
x = client.rpc_async("div", [1, 0])

In [13]:
try:
    x.get()
except Exception as e:
    print(e)

Error -32603 : Internal RPC error.

 Traceback (most recent call last):
  File "\\ntcimdavinfo2\fondos\arquitectura_gestora\desarrollo\py_distributed_processing\distributed_processing\worker.py", line 472, in process_single_request
    result = self._exec_method_in_queue(
        dispatched_to, request["method"], args, kwargs
    )
  File "\\ntcimdavinfo2\fondos\arquitectura_gestora\desarrollo\py_distributed_processing\distributed_processing\worker.py", line 394, in _exec_method_in_queue
    return funcs[method](*args, **kwargs)
           ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "C:\Users\agarcia\AppData\Local\Temp\ipykernel_52320\1531640066.py", line 8, in div
    return x / y
           ~~^~~
ZeroDivisionError: division by zero



In [14]:
#x = client.rpc_sync("div", [1, 0])

In [15]:
client.rpc_sync("add", [1, 1])

2

In [16]:
def f(x,y): return x + y

y = client.rpc_async_fn(f, [1, 2.0])

In [17]:
y.get()

3.0

In [18]:
client.rpc_sync_fn(f, [3.0, 2.0])

5.0

In [19]:
fs =[]
tp = []
N = 10
for i in range(N):
    fn = random.choice(("add", "mul", "div", "lista", "tupla", "dic"))
    t = (fn, [random.random(),random.random()], {})
    print(t)
    tp.append(t)
    fs.append(client.rpc_async(t[0], t[1]))

t = ("sleep", [10], {})
tp.append(t)
fs.append(client.rpc_async(t[0], t[1]))



('add', [0.3516222974958745, 0.4773636367846972], {})
('tupla', [0.424915760495309, 0.7123603051872257], {})
('tupla', [0.6416443259395709, 0.26496574210856216], {})
('div', [0.7587113413232894, 0.9663604444257268], {})
('add', [0.7521029571774563, 0.7646324932548614], {})
('lista', [0.010103470951384308, 0.32484597104487956], {})
('lista', [0.8691140484143584, 0.5422448732618065], {})
('tupla', [0.8070929537499016, 0.37526352427169796], {})
('mul', [0.5185408135708498, 0.6747752002551551], {})
('div', [0.335073139602154, 0.6559431184316721], {})


In [20]:
resultados = [f.get() for f in fs]
resultados

[0.8289859342805717,
 (0.424915760495309, 0.7123603051872257),
 (0.6416443259395709, 0.26496574210856216),
 0.7851225137574461,
 1.5167354504323178,
 [0.010103470951384308, 0.32484597104487956],
 [0.8691140484143584, 0.5422448732618065],
 (0.8070929537499016, 0.37526352427169796),
 0.34989848131774126,
 0.5108265186214583,
 None]

In [21]:
fs = client.rpc_multi_async(tp)

In [22]:
# AsynResult.status updates the information every time it is called.
# 'PENDING' state should be assumed as transitory. 
# If there are responses available in the queue or in the cache 
# status or pending() updates the AsyncResult object.

[f.status for f in fs]

['OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'PENDING',
 'PENDING',
 'PENDING']

In [23]:
client.wait_responses(timeout=2)

['fs_client_1:29']

In [24]:
[f.status for f in fs]

['OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'PENDING']

In [25]:
client.wait_responses()

[]

In [26]:
# AsynResult.status updates information

[f.status for f in fs]

['OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK', 'OK']

In [27]:
try:
    client.wait_responses(["kk", "qq"])
except ValueError as e:
    print(e)

wait_responses: ['kk', 'qq'] neither in responses nor in pending.


In [28]:
client._update_cache_with_all_available_responses()

In [29]:
client.pending

{}

In [30]:
[f.get() for f in fs]

[0.8289859342805717,
 (0.424915760495309, 0.7123603051872257),
 (0.6416443259395709, 0.26496574210856216),
 0.7851225137574461,
 1.5167354504323178,
 [0.010103470951384308, 0.32484597104487956],
 [0.8691140484143584, 0.5422448732618065],
 (0.8070929537499016, 0.37526352427169796),
 0.34989848131774126,
 0.5108265186214583,
 None]

In [31]:
fs = client.rpc_batch_async(tp)

In [32]:
[f.get() for f in fs]

[0.8289859342805717,
 (0.424915760495309, 0.7123603051872257),
 (0.6416443259395709, 0.26496574210856216),
 0.7851225137574461,
 1.5167354504323178,
 [0.010103470951384308, 0.32484597104487956],
 [0.8691140484143584, 0.5422448732618065],
 (0.8070929537499016, 0.37526352427169796),
 0.34989848131774126,
 0.5108265186214583,
 None]

In [33]:
client.rpc_batch_sync(tp)

[0.8289859342805717,
 (0.424915760495309, 0.7123603051872257),
 (0.6416443259395709, 0.26496574210856216),
 0.7851225137574461,
 1.5167354504323178,
 [0.010103470951384308, 0.32484597104487956],
 [0.8691140484143584, 0.5422448732618065],
 (0.8070929537499016, 0.37526352427169796),
 0.34989848131774126,
 0.5108265186214583,
 None]

In [34]:
fs =[]
tp = []
N = 10
for i in range(N):
    fn = random.choice(("add", "mul", "div", "fake"))
    t = (fn, [random.random(),random.random()], {})
    print(t)
    tp.append(t)

('fake', [0.02189547587784091, 0.0049570766159329205], {})
('add', [0.8495091869526267, 0.6717557958852868], {})
('mul', [0.37352756455719616, 0.2910279136710143], {})
('mul', [0.17708792506564042, 0.6538431994814605], {})
('mul', [0.13439707767430065, 0.6177397506590822], {})
('mul', [0.46991174216121934, 0.012010201544341403], {})
('mul', [0.24811876872047067, 0.8077557076648963], {})
('fake', [0.8419762437218836, 0.26305600921403816], {})
('mul', [0.5810891668311045, 0.48201748043322123], {})
('mul', [0.5929071166870077, 0.27845083449414876], {})


In [35]:
tp

[('fake', [0.02189547587784091, 0.0049570766159329205], {}),
 ('add', [0.8495091869526267, 0.6717557958852868], {}),
 ('mul', [0.37352756455719616, 0.2910279136710143], {}),
 ('mul', [0.17708792506564042, 0.6538431994814605], {}),
 ('mul', [0.13439707767430065, 0.6177397506590822], {}),
 ('mul', [0.46991174216121934, 0.012010201544341403], {}),
 ('mul', [0.24811876872047067, 0.8077557076648963], {}),
 ('fake', [0.8419762437218836, 0.26305600921403816], {}),
 ('mul', [0.5810891668311045, 0.48201748043322123], {}),
 ('mul', [0.5929071166870077, 0.27845083449414876], {})]

In [36]:
client.check_registry ="never"
client.set_default_queue("cola_1")

fs = client.rpc_batch_async(tp)

In [37]:
[f.safe_get() for f in fs]

[None,
 1.5212649828379134,
 0.10870694781169592,
 0.11578773551445146,
 0.08302241725183178,
 0.0056437347314088355,
 0.20041935161274654,
 None,
 0.2800951361029687,
 0.16509548141901692]

In [38]:
client.responses

{}

In [39]:
client.check_registry ="cache"

In [40]:
try:
    x = client.rpc_batch_sync(tp, timeout=5)
except Exception as e:
    print(e)

No common queue for batch request.


In [41]:
try:    
    x = client.rpc_batch_sync(tp, timeout=5)
except Exception as e:
    print(e)

No common queue for batch request.


In [42]:
client.check_registry="never"
client.set_default_queue("cola_1")

x = client.rpc_async("kk", [1, 0])

In [43]:
try:
    x.get()
except Exception as e:
    print(e)

Error -32601 : The method does not exist/is not available.

 NoneType: None



In [44]:
y = client.rpc_async("add", [1, 0])

In [45]:
y.get(5)

1

In [46]:
def f(x,y): return x + y

client.check_registry = "never" # "never" use queue "default" don't work for rpc_async_fn or rpc_sync_fn 
y = client.rpc_async_fn(f, [1, 2.0])
try:
    print(y.get())
except Exception as e:
    print(e)

Error -32601 : The method does not exist/is not available.

 NoneType: None



In [47]:
client.check_registry = "cache"
y = client.rpc_async_fn(f, [1, 2.0])

In [48]:
y.get()

3.0

In [49]:
[k for k in client.responses.keys()]

[]

In [50]:
client.clean_used()

In [51]:
[k for k in client.responses.keys()]

[]

In [52]:
resultados

[0.8289859342805717,
 (0.424915760495309, 0.7123603051872257),
 (0.6416443259395709, 0.26496574210856216),
 0.7851225137574461,
 1.5167354504323178,
 [0.010103470951384308, 0.32484597104487956],
 [0.8691140484143584, 0.5422448732618065],
 (0.8070929537499016, 0.37526352427169796),
 0.34989848131774126,
 0.5108265186214583,
 None]

In [53]:
client.rpc_sync_fn(print, ["hola"])

In [54]:
fs =[]
tp = []
N = 10
for i in range(N):
    fn = random.choice(("sleep", "sleep"))
    t = (fn, [15], {})
    print(t)
    tp.append(t)

('sleep', [15], {})
('sleep', [15], {})
('sleep', [15], {})
('sleep', [15], {})
('sleep', [15], {})
('sleep', [15], {})
('sleep', [15], {})
('sleep', [15], {})
('sleep', [15], {})
('sleep', [15], {})


In [55]:
client.check_registry="never"
fs = client.rpc_multi_async(tp, retry=True)

In [56]:
from time import time

def gather_kk(arl, timeout=None, step=5, max_dts=0):
    # Update to reset delta times
    pending_ars = [ar for ar in arl if not(ar.ok() or ar.failed())] 
    pending_ids = [ar.id for ar in pending_ars]
    if fs[0]._client.wait_responses(pending_ids, timeout=0) == []:
        return

    N = len(arl)

    max_dts = max_dts

    t_0 = time()

    i = 0
    # ok() and failed() don't update.
    # pending() updates, every time is called, all the ids 
    # that have a response available in the queue 
    pending_ars = [ar for ar in arl if not(ar.ok() or ar.failed())]
    pending_ars0= pending_ars
    pending_ids = [ar.id for ar in pending_ars]
    t_prev = time()
    n_pending_prev = len(pending_ars)
    n_pending_0 = n_pending_prev    
    while (time() - t_0) <= timeout:
        try:
            if fs[0]._client.wait_responses(pending_ids, timeout=5) == []:
                return
        except TimeoutError:
            pass
        dts = [ar.finished_time - t_0 for ar in pending_ars0 if (ar.ok() or ar.failed())]
        if len(dts)>0:
            #max_dts = max(max_dts, max(dts))
            max_dts = max(dts)    
        pending_ars = [ar for ar in arl if not(ar.ok() or ar.failed())] 
        pending_ids = [ar.id for ar in pending_ars]
        if len(pending_ars) < n_pending_prev:
            t_prev = time()
            n_pending_prev = len(pending_ars)  

        avg =  (time()-t_0)/(n_pending_0 -len(pending_ars)) if (n_pending_0 -len(pending_ars))>0 else 0

        print(f"{i}: seconds {time()-t_0}s, AR recovered {N-len(pending_ars)}, \
                AR left {len(pending_ars)}, max delta {max_dts}, avg {avg}")
        i += 0


In [57]:
gather(fs, 30, 5)

False

In [58]:
[f.status for f in fs]

['OK',
 'OK',
 'PENDING',
 'PENDING',
 'PENDING',
 'PENDING',
 'PENDING',
 'PENDING',
 'PENDING',
 'PENDING']

In [59]:
[(time() if f.finished_time is None else f.finished_time) -f.creation_time for f in fs]

[15.140183687210083,
 29.864397525787354,
 31.053590297698975,
 31.037001848220825,
 31.017271757125854,
 30.9940402507782,
 30.977222204208374,
 30.959341526031494,
 30.94121217727661,
 30.923853158950806]

In [60]:
[f.finished_time for f in fs]

[1784133089.35977,
 1784133104.385689,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [61]:
client.pending

{'fs_client_1:69': 1784133074.539856,
 'fs_client_1:70': 1784133074.5564451,
 'fs_client_1:71': 1784133074.5761747,
 'fs_client_1:72': 1784133074.5994065,
 'fs_client_1:73': 1784133074.616226,
 'fs_client_1:74': 1784133074.6341057,
 'fs_client_1:75': 1784133074.6522357,
 'fs_client_1:76': 1784133074.6695943}

In [62]:
fs[3].status

'PENDING'

In [63]:
fs[4].retry()

True

In [64]:
# [f.retry() for f in fs if not f.done()]
# no hace falta chequear si está pendiente.
[f.retry() for f in fs]

[False, False, True, True, True, True, True, True, True, True]

In [65]:
client.check_registry="always"
client.all_queues_for_method("info")

['fs_server_1']

In [66]:
client.connector.all_queues_for_method("info")

['requests_fs_server_1']

In [67]:
client.update_registry_cache()

In [68]:
client.check_registry="Never"
client.all_queues_for_method("hola")

['cola_1']

In [69]:
client.check_registry="always"
a = client.rpc_async("info")

In [70]:
client.rpc_sync("div", [2,1], timeout=10)

TimeoutError: 

In [ ]:
x = a.get()

In [ ]:
client.registry

In [ ]:
def registry_one_step(x):
    worker_id = x[0]
    registry = {}
    for queue, methods in x[1].items():
        for method in methods:
            if method in registry:
                registry[method].append(queue)
            else:
                registry[method] = [queue]
    return registry 

def queues_workers_one_step(x):
    worker_id = x[0]
    registry = {}
    for queue in x[1]:
        if queue in registry:
            registry[queue].append(worker_id)
        else:
            registry[queue] = [worker_id]
    return registry 

def all_workers_for_queue():
    # all requests queues for "info" method
    # One queue per worker

    rr = {}
    qq = {}
    all_worker_queues = client.connector.all_queues_for_method("info")
    for worker in all_worker_queues:
        try:
            x = client.rpc_sync("info", timeout=10)
        except TimeoutError:
            x = None
        
        if x is not None:
            for method, qs in registry_one_step(x).items():
                if method not in rr:
                    rr[method] = qs
                else:
                    rr[method] += qs
            for q, ws in queues_workers_one_step(x).items():
                if q not in qq:
                    qq[q] = ws
                else:
                    qq[q] += ws
    return rr, qq
            

In [ ]:
registry_one_step(x)

In [ ]:
queues_workers_one_step(x)

In [ ]:
r, q = all_workers_for_queue()

In [ ]:
r

In [ ]:
q

In [ ]:
t = (1,2,3,4)

In [ ]:
t[:6]

In [ ]:
t[:5]